# Transfer of English norms

Braginsky et al. attach the English ratings to every language through the uni_lemma, which assumes a rating belongs to the concept rather than the English word. Here we test that for the four languages where NoRaRe has ratings of their own.

English is compared with a second English study, the Glasgow Norms. That shows how well two independent rating studies agree on the same words, and gives the other languages something to be measured against.

In [ ]:
import os
import sys

# notebooks lie one level down, src is imported from the project root
if "src" not in os.listdir():
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.aoa import items
from src.common import schema
from src.common.config import FIGURES, TABLES
from src.norms import norare

In [ ]:
LANGUAGES = ["ita", "fra", "spa", "nld", "eng"]

frames = []
for language in LANGUAGES:
    ours = schema.read(language)
    words = items.load(language)
    words = words[words["uni_lemma"].isin(ours["uni_lemma"])]
    native = norare.load(words, language)
    # for English, "native" is the Glasgow Norms
    frames.append(ours.merge(native, on="uni_lemma", suffixes=("_eng", "_native")))

both = pd.concat(frames)
both.groupby(["language", "lexical_category"]).size().unstack()

In [ ]:
both[(both["language"] == "nld") & (both["lexical_category"] == "nouns")]

Verheyen et al. rated adjectives only, so a Dutch noun should not be here. It is the body part *arm*, matched to the Dutch adjective *arm* (poor) because the two are spelled alike.

## Agreement

Correlations per language, norm and lexical category, and once over all words. The regressions run per category, so the correlation within a category is the one that matters. The pooled one is there to show how much it differs.

Pearson's r comes with a 95% interval from the Fisher transformation. Spearman is added because English concreteness bunches at the top of the scale for nouns. The scales differ between studies, but a correlation does not depend on scale.

In [ ]:
def agreement(pairs):
    english, native = pairs.iloc[:, 0], pairs.iloc[:, 1]
    n = len(pairs)
    r = english.corr(native)
    # Fisher z interval, it gets wide quickly below 30 words
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    return {
        "n": n,
        "r": r,
        "low": np.tanh(z - 1.96 * se),
        "high": np.tanh(z + 1.96 * se),
        "spearman": english.corr(native, method="spearman"),
    }


groups = list(both.groupby(["language", "lexical_category"]))
groups += [((language, "all"), group) for language, group in both.groupby("language")]

rows = []
for (language, category), group in groups:
    for norm in norare.NORMS:
        pairs = group[[f"{norm}_eng", f"{norm}_native"]].dropna()
        # with fewer than five words there is no correlation worth reporting
        if len(pairs) < 5:
            continue
        rows.append({"language": language, "category": category, "norm": norm, **agreement(pairs)})

table = pd.DataFrame(rows)

TABLES.mkdir(parents=True, exist_ok=True)
table.to_csv(TABLES / "norm_transfer.csv", index=False)
table.round(2)

In [ ]:
table.pivot_table(index=["norm", "category"], columns="language", values="r").round(2)[LANGUAGES]

## Scatter plots

Both ratings are standardised within each language, so the axes are comparable across studies. Colours mark the lexical category.

In [ ]:
def z(values):
    return (values - values.mean()) / values.std()


# fixed colours, otherwise English with its extra function words shifts them
COLOURS = {"nouns": "C0", "predicates": "C1", "function_words": "C2"}

fig, axes = plt.subplots(3, 5, figsize=(15, 9), sharex=True, sharey=True)

for row, norm in enumerate(norare.NORMS):
    for col, language in enumerate(LANGUAGES):
        ax = axes[row, col]
        data = both[both["language"] == language].dropna(subset=[f"{norm}_eng", f"{norm}_native"])
        # standardised within the language, not the category, so the gap between categories stays visible
        data = data.assign(x=z(data[f"{norm}_eng"]), y=z(data[f"{norm}_native"]))

        for category, group in data.groupby("lexical_category"):
            ax.scatter(group["x"], group["y"], s=12, alpha=0.6, color=COLOURS[category], label=category)

        if row == 0:
            ax.set_title("eng vs Glasgow" if language == "eng" else language)
        if col == 0:
            ax.set_ylabel(f"{norm}\nnative (z)")
        if row == 2:
            ax.set_xlabel("English (z)")

# the English panel is the only one with all three categories
axes[0, 4].legend(fontsize=8)

FIGURES.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURES / "norm_transfer.png", dpi=150, bbox_inches="tight")
plt.show()

## What transfers

Valence transfers well. Within nouns the correlations lie between 0.64 and 0.84, within predicates between 0.94 and 0.99. The two English studies agree about as closely, with 0.75 and 0.96.

Arousal is weaker, but so is the English baseline at 0.63 for nouns and 0.53 for predicates. Italian and Spanish reach that level. French stays clearly below it at 0.35 and 0.37.

Concreteness looks fine over all words, between 0.70 and 0.87 for Italian, French and Spanish. Within nouns it drops to between 0.04 and 0.29, while the two English studies still reach 0.68. The plots show why: the nouns form a narrow vertical band, so English gives them almost the same value while the other languages spread them out. The high overall correlation comes from nouns being more concrete than predicates, which both sides agree on. The regressions run per category, so for nouns the English concreteness rating carries little of what native speakers judge.

Dutch has 20 predicates and the false match above, Spanish only 5 predicates. Their intervals are too wide to read much into.